# PINN a posteriori estimator on the unit square

This notebook trains a PINN for a manufactured Poisson problem on $\Omega=[0,1]^2$, then reports:

- empirical diagnostics from point samples,
- **certified interval enclosures** for a boundary-trace $L^2(\partial\Omega)$ mismatch,
- a **certified interval enclosure** for an interval-compatible residual surrogate $\widehat r_\theta$,
- and a combined practical indicator.

> **Important rigor note:** with current intervalNets primitives, direct certified propagation of second derivatives (Laplacian) is not available for the full PINN. We therefore separate the true PDE residual (empirical) from a certified surrogate residual model (interval-certified).

## 1) Setup and imports

In [ ]:
from __future__ import annotations

import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# If running from repo checkout without editable install, add src/.
repo_root = Path.cwd()
while not (repo_root / "src" / "intervalnets").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root / "src") not in __import__('sys').path:
    __import__('sys').path.insert(0, str(repo_root / "src"))

from intervalnets import Interval, IntervalTensor, enable_interval_eval

enable_interval_eval(enclosure_mode="slope")

dtype = torch.float64
device = torch.device("cpu")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Using torch={torch.__version__}, dtype={dtype}, seed={SEED}")

## 2) Problem definition (PDE, exact solution, forcing, BC)

In [ ]:
PI = math.pi

def u_exact(xy: torch.Tensor) -> torch.Tensor:
    x = xy[:, 0:1]
    y = xy[:, 1:2]
    return torch.sin(PI * x) * torch.sin(PI * y)


def forcing_f(xy: torch.Tensor) -> torch.Tensor:
    x = xy[:, 0:1]
    y = xy[:, 1:2]
    return 2.0 * (PI ** 2) * torch.sin(PI * x) * torch.sin(PI * y)


def g_boundary(xy: torch.Tensor) -> torch.Tensor:
    # For this manufactured problem, g = 0 on the full boundary.
    return torch.zeros((xy.shape[0], 1), dtype=xy.dtype, device=xy.device)

print("Problem:")
print("- Domain Ω = [0,1]^2")
print("- PDE    -Δu = f")
print("- BC     u = g on ∂Ω")
print("- Exact  u*(x,y)=sin(πx)sin(πy), so f=2π^2 sin(πx)sin(πy), g=0 on ∂Ω")

## 3) PINN model

In [ ]:
def make_pinn(width: int = 64, hidden_layers: int = 4) -> nn.Sequential:
    layers: list[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
    for _ in range(hidden_layers - 1):
        layers += [nn.Linear(width, width), nn.Tanh()]
    layers += [nn.Linear(width, 1)]
    model = nn.Sequential(*layers)
    return model

model = make_pinn(width=64, hidden_layers=4).to(device=device, dtype=dtype)
print(model)

## 4) Training data sampling (interior + boundary)

In [ ]:
def sample_interior(n: int) -> torch.Tensor:
    # Uniform random points in [0,1]^2.
    return torch.rand((n, 2), dtype=dtype, device=device)


def sample_boundary(n_per_edge: int) -> torch.Tensor:
    t = torch.rand((n_per_edge, 1), dtype=dtype, device=device)
    zeros = torch.zeros_like(t)
    ones = torch.ones_like(t)
    e1 = torch.cat([t, zeros], dim=1)  # (t,0)
    e2 = torch.cat([t, ones], dim=1)   # (t,1)
    e3 = torch.cat([zeros, t], dim=1)  # (0,t)
    e4 = torch.cat([ones, t], dim=1)   # (1,t)
    return torch.cat([e1, e2, e3, e4], dim=0)


def laplacian_u(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    xy_req = xy.detach().clone().requires_grad_(True)
    u = model(xy_req)
    grad_u = torch.autograd.grad(
        u, xy_req, grad_outputs=torch.ones_like(u), create_graph=True
    )[0]
    u_x = grad_u[:, 0:1]
    u_y = grad_u[:, 1:2]
    u_xx = torch.autograd.grad(
        u_x, xy_req, grad_outputs=torch.ones_like(u_x), create_graph=True
    )[0][:, 0:1]
    u_yy = torch.autograd.grad(
        u_y, xy_req, grad_outputs=torch.ones_like(u_y), create_graph=True
    )[0][:, 1:2]
    return u_xx + u_yy


def residual_r(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    # r_theta = -Δu_theta - f
    return -laplacian_u(model, xy) - forcing_f(xy)

# Training sample sizes
N_INTERIOR = 1024
N_BDRY_PER_EDGE = 256

## 5) Training loop

In [ ]:
EPOCHS = 1200
LR = 1e-3
W_INTERIOR = 1.0
W_BOUNDARY = 20.0  # emphasize Dirichlet condition

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"loss_total": [], "loss_interior": [], "loss_boundary": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    xy_i = sample_interior(N_INTERIOR)
    xy_b = sample_boundary(N_BDRY_PER_EDGE)

    r_i = residual_r(model, xy_i)
    b_mismatch = model(xy_b) - g_boundary(xy_b)

    loss_interior = torch.mean(r_i ** 2)
    loss_boundary = torch.mean(b_mismatch ** 2)
    loss = W_INTERIOR * loss_interior + W_BOUNDARY * loss_boundary

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    history["loss_total"].append(float(loss.detach().cpu()))
    history["loss_interior"].append(float(loss_interior.detach().cpu()))
    history["loss_boundary"].append(float(loss_boundary.detach().cpu()))

    if epoch % 200 == 0:
        print(
            f"epoch={epoch:4d} | total={history['loss_total'][-1]:.3e} | "
            f"interior={history['loss_interior'][-1]:.3e} | boundary={history['loss_boundary'][-1]:.3e}"
        )

model.eval();

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.semilogy(history["loss_total"], label="total")
ax.semilogy(history["loss_interior"], label="interior (residual MSE)")
ax.semilogy(history["loss_boundary"], label="boundary (MSE)")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("PINN training curves")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 6) Sanity diagnostics (sampled residual and boundary mismatch)

In [ ]:
@torch.no_grad()
def sample_l2_norm(values: torch.Tensor) -> float:
    return float(torch.sqrt(torch.mean(values**2)).cpu())

# Dense random diagnostics
xy_i_diag = sample_interior(20000)
xy_b_diag = sample_boundary(2000)

r_diag = residual_r(model, xy_i_diag).detach()
b_diag = (model(xy_b_diag) - g_boundary(xy_b_diag)).detach()
u_diag_err = (model(xy_i_diag) - u_exact(xy_i_diag)).detach()

empirical = {
    "||r_theta||_{L2(Ω), MC}": sample_l2_norm(r_diag),
    "||u_theta-g||_{L2(∂Ω), MC-edge-uniform}": sample_l2_norm(b_diag),
    "||u_theta-u*||_{L2(Ω), MC}": sample_l2_norm(u_diag_err),
}

for k, v in empirical.items():
    print(f"{k:45s}: {v:.6e}")

## 7) Certified interval enclosure of residual $L^2(\Omega)$

### Rigorous-status note

The true residual is $r_\theta(x,y) = -\Delta u_\theta(x,y)-f(x,y)$. Direct **certified** interval propagation of second derivatives ($\Delta u_\theta$) is currently not available in intervalNets APIs used here (`eval`, `eval_jacobian`, `lpnorm`, `sobolev_norm` are first-derivative level).

So we build an interval-compatible surrogate network $\widehat r_\theta$ trained on many autograd residual samples of $r_\theta$, then certify $\|\widehat r_\theta\|_{L^2(\Omega)}$ via `lpnorm`. This part is certified for the surrogate, while closeness to true residual remains empirical.

In [ ]:
# Build residual training data from the trained PINN
model.eval()

N_RESIDUAL_FIT = 30000
xy_res_fit = sample_interior(N_RESIDUAL_FIT)
with torch.enable_grad():
    r_targets = residual_r(model, xy_res_fit).detach()

residual_model = nn.Sequential(
    nn.Linear(2, 64), nn.Tanh(),
    nn.Linear(64, 64), nn.Tanh(),
    nn.Linear(64, 64), nn.Tanh(),
    nn.Linear(64, 1),
).to(device=device, dtype=dtype)

opt_r = torch.optim.Adam(residual_model.parameters(), lr=1e-3)
BATCH = 2048
EPOCHS_RESIDUAL_FIT = 500
res_fit_losses = []

for epoch in range(1, EPOCHS_RESIDUAL_FIT + 1):
    perm = torch.randperm(N_RESIDUAL_FIT, device=device)
    epoch_loss = 0.0
    for i in range(0, N_RESIDUAL_FIT, BATCH):
        idx = perm[i:i+BATCH]
        pred = residual_model(xy_res_fit[idx])
        loss = torch.mean((pred - r_targets[idx]) ** 2)
        opt_r.zero_grad()
        loss.backward()
        opt_r.step()
        epoch_loss += float(loss.detach().cpu())
    res_fit_losses.append(epoch_loss)

residual_model.eval()
print(f"Residual surrogate fit final epoch-loss: {res_fit_losses[-1]:.3e}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
ax.semilogy(res_fit_losses)
ax.set_title("Residual surrogate training loss")
ax.set_xlabel("epoch")
ax.set_ylabel("sum mini-batch MSE")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Certified L2 enclosure for surrogate residual norm
DOMAIN_2D = IntervalTensor.from_bounds([0.0, 0.0], [1.0, 1.0])

RESIDUAL_NORM_ITERS = 4
RESIDUAL_NORM_THETA = 0.6
RESIDUAL_FORWARD_SPLITS = 2

residual_l2_iv = residual_model.lpnorm(
    DOMAIN_2D,
    p=2.0,
    iterations=RESIDUAL_NORM_ITERS,
    theta=RESIDUAL_NORM_THETA,
    forward_refine_splits=RESIDUAL_FORWARD_SPLITS,
)

print(
    f"Certified surrogate ||r_hat||_L2(Ω) ∈ [{float(residual_l2_iv.lower):.6e}, {float(residual_l2_iv.upper):.6e}]"
)
print(f"Interval width: {float(residual_l2_iv.upper - residual_l2_iv.lower):.6e}")

# Empirical surrogate-vs-true residual mismatch
xy_val = sample_interior(20000)
with torch.enable_grad():
    r_true_val = residual_r(model, xy_val).detach()
with torch.no_grad():
    r_hat_val = residual_model(xy_val)

surrogate_fit_l2 = sample_l2_norm(r_hat_val - r_true_val)
print(f"Empirical ||r_hat - r_true||_L2(Ω) (MC): {surrogate_fit_l2:.6e}")

## 8) Certified interval enclosure of boundary trace $L^2(\partial\Omega)$

In [ ]:
def make_edge_map(kind: str) -> nn.Linear:
    # Linear map t -> (x,y) for each edge; weights are fixed and exact constants.
    layer = nn.Linear(1, 2, bias=True).to(device=device, dtype=dtype)
    with torch.no_grad():
        if kind == "t0":       # (t,0)
            layer.weight[:] = torch.tensor([[1.0], [0.0]], dtype=dtype)
            layer.bias[:] = torch.tensor([0.0, 0.0], dtype=dtype)
        elif kind == "t1":     # (t,1)
            layer.weight[:] = torch.tensor([[1.0], [0.0]], dtype=dtype)
            layer.bias[:] = torch.tensor([0.0, 1.0], dtype=dtype)
        elif kind == "0t":     # (0,t)
            layer.weight[:] = torch.tensor([[0.0], [1.0]], dtype=dtype)
            layer.bias[:] = torch.tensor([0.0, 0.0], dtype=dtype)
        elif kind == "1t":     # (1,t)
            layer.weight[:] = torch.tensor([[0.0], [1.0]], dtype=dtype)
            layer.bias[:] = torch.tensor([1.0, 0.0], dtype=dtype)
        else:
            raise ValueError(kind)
    for p in layer.parameters():
        p.requires_grad_(False)
    return layer

edge_models = {
    "edge1:(t,0)": nn.Sequential(make_edge_map("t0"), model),
    "edge2:(t,1)": nn.Sequential(make_edge_map("t1"), model),
    "edge3:(0,t)": nn.Sequential(make_edge_map("0t"), model),
    "edge4:(1,t)": nn.Sequential(make_edge_map("1t"), model),
}

DOMAIN_1D = IntervalTensor.from_bounds([0.0], [1.0])
BND_ITERS = 5
BND_THETA = 0.6
BND_FORWARD_SPLITS = 3

edge_norm_intervals: dict[str, Interval] = {}
for name, edge_model in edge_models.items():
    iv = edge_model.lpnorm(
        DOMAIN_1D,
        p=2.0,
        iterations=BND_ITERS,
        theta=BND_THETA,
        forward_refine_splits=BND_FORWARD_SPLITS,
    )
    edge_norm_intervals[name] = iv
    print(f"{name:14s}: [{float(iv.lower):.6e}, {float(iv.upper):.6e}] (width={float(iv.upper-iv.lower):.3e})")

# Rigorous aggregate over the four edges:
# ||u-g||_{L2(∂Ω)} = sqrt(sum_k ||b_k||_{L2(0,1)}^2)
sum_lower = 0.0
sum_upper = 0.0
for iv in edge_norm_intervals.values():
    lk = max(0.0, float(iv.lower))
    uk = max(0.0, float(iv.upper))
    sum_lower += lk * lk
    sum_upper += uk * uk

boundary_l2_iv = Interval.from_bounds(math.sqrt(sum_lower), math.sqrt(sum_upper))
print(
    f"Global certified ||u_theta-g||_L2(∂Ω) ∈ [{float(boundary_l2_iv.lower):.6e}, {float(boundary_l2_iv.upper):.6e}]"
)
print(f"Global boundary interval width: {float(boundary_l2_iv.upper-boundary_l2_iv.lower):.6e}")

## 9) Combined a posteriori estimator

In [ ]:
# Practical estimator: eta = ||r|| + ||trace mismatch||
# Here: certified interval for surrogate residual + certified interval for boundary mismatch.
eta_iv = residual_l2_iv + boundary_l2_iv
print(f"η interval (surrogate-certified): [{float(eta_iv.lower):.6e}, {float(eta_iv.upper):.6e}]")
print(f"η interval width: {float(eta_iv.upper-eta_iv.lower):.6e}")

## 10) Summary table and conclusions

In [ ]:
def iv_row(name: str, iv: Interval, empirical_value: float | None = None) -> dict:
    return {
        "metric": name,
        "empirical (MC/sample)": np.nan if empirical_value is None else empirical_value,
        "certified lower": float(iv.lower),
        "certified upper": float(iv.upper),
        "interval width": float(iv.upper - iv.lower),
    }

rows = []
rows.append(
    iv_row(
        "Residual norm (surrogate) ||r_hat||_L2(Ω)",
        residual_l2_iv,
        empirical_value=empirical["||r_theta||_{L2(Ω), MC}"],
    )
)
rows.append(
    iv_row(
        "Boundary mismatch ||u-g||_L2(∂Ω)",
        boundary_l2_iv,
        empirical_value=empirical["||u_theta-g||_{L2(∂Ω), MC-edge-uniform}"],
    )
)
rows.append(iv_row("Combined η = residual + boundary", eta_iv, empirical_value=None))

for name, iv in edge_norm_intervals.items():
    rows.append(iv_row(f"Per-edge {name}", iv, empirical_value=None))

summary_df = pd.DataFrame(rows)
summary_df

### Interpretation

- The PINN training drives sampled interior residual and boundary mismatch down.
- The boundary trace term is certified directly for the trained PINN through interval propagation on each edge map.
- The interior residual term is **certified for the surrogate residual model** $\widehat r_\theta$ (not directly for $r_\theta$) due to unavailable second-derivative interval primitives in the current API.
- Therefore, the final reported $\eta_\theta$ is a practical mixed estimator: certified for boundary and surrogate-residual components, with empirical checks for the true residual and surrogate fit quality.